# 新客群消贷频繁支用概率预测

使用 `kechuang_merged0729.csv` 构造 `y_freq` 标签并训练 Easy Ensemble + LightGBM，可对客户级白名单或一客多贷的申贷未支用数据预测。

核心口径：
- `y_freq`：`ac_curr_bal_diff >= P80` 且 `ba_out_bal_diff >= P80`；
- Easy Ensemble：10 个随机欠采样子集，每个子集正负样本 1:1；
- 自动识别数据结构：白名单使用 `pre_credit_limit`；申贷未支用数据使用 `credamt` 并执行一客多贷聚合；
- 输出仅包含客户 ID 和预测概率，按预测概率从高到低排序。

## 1. 参数配置

In [ ]:
from pathlib import Path

# auto 会根据字段自动判断：loanacctno+credamt=申贷未支用一客多贷；pre_credit_limit=客户级白名单。
PREDICTION_COHORT_MODE = globals().get('PREDICTION_COHORT_MODE_OVERRIDE', 'auto')
TRAIN_FILE = Path(globals().get('TRAIN_FILE_OVERRIDE', 'kechuang_merged0729.csv'))
PREDICT_FILE = Path(globals().get('PREDICT_FILE_OVERRIDE', 'whitelist0810.csv'))
OUTPUT_FILE = Path(globals().get('OUTPUT_FILE_OVERRIDE', 'whitelist0810_y_freq_probability.csv'))
CSV_ENCODING = 'utf-8-sig'

TRAIN_SNAPSHOT_DATE = '2026-06-24'
PREDICT_SNAPSHOT_DATE = '2026-07-31'
Y_FREQ_MODE = 'curr_p80_and_bout_p80'
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
EARLY_STOPPING_ROUNDS = 50
EASY_ENSEMBLE_N_ESTIMATORS = 10
EASY_ENSEMBLE_RATIO = 1.0  # 每个子集多数类/少数类=1，即正负1:1

ADD_QUOTA_SQ = False
ADD_QUOTA_CUBE = False
ADD_QUOTA_LOG = False
CATEGORICAL_CANDIDATES = [
    'gnd_cd', 'mar_sttn_cd', 'education_cd', 'occup_cd', 'cst_star_cd', 'busikind'
]
LGB_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'n_estimators': 500,
    'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': -1,
    'min_child_samples': 20, 'subsample': 0.8, 'bagging_freq': 1,
    'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'random_state': RANDOM_STATE, 'verbose': -1,
}
print('训练数据:', TRAIN_FILE.resolve())
print('预测数据:', PREDICT_FILE.resolve())
print('输出名单:', OUTPUT_FILE.resolve())

## 2. 导入与分类任务相同的清洗、采样和建模组件

In [ ]:
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# 支持 notebook 位于当前额度优化目录或分类预测目录。
module_candidates = [
    Path.cwd(),
    Path.cwd().parent / '2-分类预测任务-0728修改版',
]
module_dir = next((p for p in module_candidates if (p / 'load_kechuang_potential_data.py').exists()), None)
if module_dir is None:
    raise FileNotFoundError('未找到 load_kechuang_potential_data.py，请将 notebook 放在项目目录结构内运行。')
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from load_kechuang_potential_data import (
    load_kechuang_potential, read_data, rename_kechuang_cols, cast_cat_cols,
    aggregate_by_customer_potential, clean_data_potential, drop_post_label_cols,
    PotentialFeaturePreprocessor,
)
from sampling_methods import sampler_factory, print_sampling_summary
print('清洗模块:', module_dir / 'load_kechuang_potential_data.py')
print('Easy Ensemble 采样模块:', Path.cwd() / 'sampling_methods.py')

def read_prediction_data_robust(file_path):
    """读取预测数据：UTF-8失败后尝试中文编码，最后替换无法解码的异常字符。"""
    path = Path(file_path)
    suffix = path.suffix.lower()
    if suffix in ('.xlsx', '.xls'):
        data = pd.read_excel(path)
        print(f'预测文件读取方式：Excel；{len(data):,} 行 × {len(data.columns):,} 列')
        return data
    if suffix not in ('.csv', '.txt'):
        raise ValueError(f'预测文件仅支持 CSV/TXT/Excel，当前后缀：{suffix}')

    decode_errors = []
    for encoding in ('utf-8-sig', 'gb18030', 'gbk'):
        try:
            data = pd.read_csv(path, encoding=encoding)
            print(f'预测文件读取编码：{encoding}；{len(data):,} 行 × {len(data.columns):,} 列')
            return data
        except UnicodeDecodeError as exc:
            decode_errors.append(f'{encoding}: {exc}')

    # 极少数文件混入多种编码或损坏字节时，保留整行结构并用 U+FFFD 替换异常字符。
    with open(path, 'r', encoding='utf-8-sig', errors='replace', newline='') as handle:
        data = pd.read_csv(handle)
    replacement_count = int(sum(
        data[col].astype('string').str.count('�').fillna(0).sum()
        for col in data.select_dtypes(include=['object', 'string']).columns
    ))
    print('[警告] UTF-8/GB18030/GBK 严格解码均失败，已用 UTF-8 errors=replace 容错读取。')
    print(f'异常字符替换数量：{replacement_count:,}；请重点核对包含替换字符的数据字段。')
    print('严格解码错误摘要：', decode_errors)
    return data

## 3. 加载全部科创人才数据并构造频繁支用标签

In [ ]:
if not TRAIN_FILE.exists():
    raise FileNotFoundError(f'未找到训练文件：{TRAIN_FILE.resolve()}')

(train_clean, _, _, _, y_all, _, thresholds, _, _) = load_kechuang_potential(
    file_path=str(TRAIN_FILE),
    snapshot_date=TRAIN_SNAPSHOT_DATE,
    target='y_freq',
    csv_encoding=CSV_ENCODING,
    add_quota_sq=ADD_QUOTA_SQ,
    add_quota_cube=ADD_QUOTA_CUBE,
    add_quota_log=ADD_QUOTA_LOG,
    apply_maturity_filter=False,
    y_freq_mode=Y_FREQ_MODE,
    apply_eff_date_filter=False,
    dedup_cst_loan=False,
    exclude_dq_start_customers=True,
    label_threshold_train_ratio=0.60,
    require_dual_label_cohort=True,
)
y_all = train_clean['y_freq'].astype(int)
if y_all.nunique() != 2:
    raise ValueError(f'y_freq 必须同时包含0和1，当前类别：{sorted(y_all.unique())}')
print(f'训练客户数：{len(train_clean):,}')
print(f'y_freq 正样本：{int(y_all.sum()):,}，正样本率：{y_all.mean():.4%}')
print('标签阈值:', thresholds)

## 4. 用内部验证集确定迭代轮数

内部验证集只用于 LightGBM 早停。确定每个 Easy Ensemble 子模型的轮数后，会在全部训练客户上重新拟合最终模型。

In [ ]:
train_idx, valid_idx = train_test_split(
    np.arange(len(train_clean)), test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE, stratify=y_all.to_numpy(),
)
train_part = train_clean.iloc[train_idx].copy()
valid_part = train_clean.iloc[valid_idx].copy()
y_train_part = train_part['y_freq'].astype(int)
y_valid_part = valid_part['y_freq'].astype(int)

round_preprocessor = PotentialFeaturePreprocessor(
    target='y_freq', add_quota_sq=ADD_QUOTA_SQ, add_quota_cube=ADD_QUOTA_CUBE,
    add_quota_log=ADD_QUOTA_LOG, categorical_features=CATEGORICAL_CANDIDATES,
).fit(train_part)
X_train_part = round_preprocessor.transform(train_part)
X_valid_part = round_preprocessor.transform(valid_part)
round_cat_features = [c for c in round_preprocessor.categorical_features_ if c in X_train_part.columns]

round_sampler = sampler_factory(
    method='easy_ensemble', random_state=RANDOM_STATE,
    n_estimators=EASY_ENSEMBLE_N_ESTIMATORS, ratio=EASY_ENSEMBLE_RATIO,
)
round_subsets = round_sampler.fit_resample(X_train_part, y_train_part)
best_rounds = []
valid_probabilities = []
params = LGB_PARAMS.copy()
max_rounds = int(params.pop('n_estimators'))
for model_no, (X_sub, y_sub) in enumerate(round_subsets, start=1):
    print_sampling_summary(y_train_part, y_sub, f'Easy Ensemble 子模型{model_no}')
    dtrain = lgb.Dataset(X_sub, label=y_sub, feature_name=list(X_sub.columns), categorical_feature=round_cat_features)
    dvalid = lgb.Dataset(X_valid_part, label=y_valid_part, reference=dtrain, feature_name=list(X_sub.columns), categorical_feature=round_cat_features)
    model = lgb.train(
        params, dtrain, num_boost_round=max_rounds, valid_sets=[dvalid],
        valid_names=['validation'], early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=False,
    )
    rounds = int(model.best_iteration or max_rounds)
    best_rounds.append(rounds)
    valid_probabilities.append(model.predict(X_valid_part, num_iteration=rounds))
    print(f'  子模型{model_no}: 最优迭代轮数={rounds}')

validation_probability = np.mean(np.vstack(valid_probabilities), axis=0)
print(f'内部验证集 AUC={roc_auc_score(y_valid_part, validation_probability):.4f}')
print(f'内部验证集 AUPRC={average_precision_score(y_valid_part, validation_probability):.4f}')
print('最终模型固定迭代轮数:', best_rounds)

## 5. 清洗客户级白名单并严格对齐训练特征

In [ ]:
if not PREDICT_FILE.exists():
    raise FileNotFoundError(f'未找到预测文件：{PREDICT_FILE.resolve()}')

prediction_raw = read_prediction_data_robust(PREDICT_FILE)
prediction_raw = rename_kechuang_cols(prediction_raw)
prediction_raw = cast_cat_cols(prediction_raw)
if 'cst_id' not in prediction_raw.columns:
    raise KeyError('预测数据中未找到客户ID字段 cst_id。')
raw_customer_id = prediction_raw['cst_id'].astype('string').str.strip()
if raw_customer_id.isna().any() or raw_customer_id.eq('').any():
    raise ValueError('预测数据存在空客户ID，请先处理。')

_resolved_prediction_mode = PREDICTION_COHORT_MODE
if _resolved_prediction_mode == 'auto':
    if {'loanacctno', 'credamt', 'rt_acct_stat_2'}.issubset(prediction_raw.columns):
        _resolved_prediction_mode = 'outstanding_unused'
    elif 'pre_credit_limit' in prediction_raw.columns:
        _resolved_prediction_mode = 'whitelist'
    else:
        raise ValueError('无法自动判断预测数据类型：申贷未支用数据应含 loanacctno/credamt/rt_acct_stat_2；白名单应含 pre_credit_limit。')
print(f'实际采用的预测数据模式：{_resolved_prediction_mode}')

if _resolved_prediction_mode == 'whitelist':
    if raw_customer_id.duplicated().any():
        examples = raw_customer_id[raw_customer_id.duplicated(keep=False)].head(10).tolist()
        raise ValueError(f'白名单应为客户级数据，但发现重复客户ID，例如：{examples}')
    if 'pre_credit_limit' not in prediction_raw.columns:
        raise KeyError('白名单缺少预授信额度字段 pre_credit_limit。')
    prediction_raw['credamt'] = pd.to_numeric(prediction_raw['pre_credit_limit'], errors='coerce')
    if prediction_raw['credamt'].isna().any():
        bad_count = int(prediction_raw['credamt'].isna().sum())
        raise ValueError(f'pre_credit_limit 有 {bad_count:,} 条为空或无法转换为数值。')
    prediction_customer = prediction_raw.copy()
elif _resolved_prediction_mode == 'outstanding_unused':
    required_columns = {'loanacctno', 'credamt', 'rt_acct_stat_2'}
    missing_required = sorted(required_columns - set(prediction_raw.columns))
    if missing_required:
        raise KeyError(f'申贷未支用数据缺少SQL约定字段：{missing_required}')
    prediction_raw['credamt'] = pd.to_numeric(prediction_raw['credamt'], errors='coerce')
    if prediction_raw['credamt'].isna().any():
        raise ValueError(f'credamt 有 {int(prediction_raw["credamt"].isna().sum()):,} 条为空或非数值。')
    print(f'一客多贷聚合前：{len(prediction_raw):,} 行，{prediction_raw["cst_id"].nunique():,} 位客户')
    # SQL是一客多贷结构，与训练数据采用相同聚合函数压成一客一行；客户授信额度按合格贷款求和。
    prediction_customer = aggregate_by_customer_potential(prediction_raw)
    if prediction_customer['cst_id'].duplicated().any():
        raise AssertionError('一客多贷聚合后客户ID仍有重复。')
    print(f'一客多贷聚合后：{len(prediction_customer):,} 行，{prediction_customer["cst_id"].nunique():,} 位客户')
    print('额度字段：使用SQL中的 credamt；同一客户有效贷款额度按训练流程求和。')
else:
    raise ValueError(f'未知 PREDICTION_COHORT_MODE：{_resolved_prediction_mode}')

customer_id = prediction_customer['cst_id'].astype('string').str.strip().reset_index(drop=True)
whitelist_clean = clean_data_potential(
    prediction_customer, snapshot_date=PREDICT_SNAPSHOT_DATE, target='y_freq'
)
whitelist_clean = drop_post_label_cols(whitelist_clean, target='y_freq')
whitelist_clean = whitelist_clean.reset_index(drop=True)

final_preprocessor = PotentialFeaturePreprocessor(
    target='y_freq', add_quota_sq=ADD_QUOTA_SQ, add_quota_cube=ADD_QUOTA_CUBE,
    add_quota_log=ADD_QUOTA_LOG, categorical_features=CATEGORICAL_CANDIDATES,
).fit(train_clean)
X_all = final_preprocessor.transform(train_clean)
X_whitelist = final_preprocessor.transform(whitelist_clean)
if list(X_all.columns) != list(X_whitelist.columns):
    raise AssertionError('训练集与白名单的模型特征未严格对齐。')
missing_source_features = [c for c in final_preprocessor.feature_names_ if c not in whitelist_clean.columns]
print(f'白名单客户数：{len(whitelist_clean):,}')
print(f'最终模型特征数：{X_all.shape[1]:,}')
print(f'白名单原始数据未提供、将使用训练集填充值的特征数：{len(missing_source_features):,}')
if missing_source_features:
    print('缺少特征:', missing_source_features)

## 6. 在全部训练客户上重训 Easy Ensemble 并输出预测名单

In [ ]:
final_sampler = sampler_factory(
    method='easy_ensemble', random_state=RANDOM_STATE,
    n_estimators=EASY_ENSEMBLE_N_ESTIMATORS, ratio=EASY_ENSEMBLE_RATIO,
)
final_subsets = final_sampler.fit_resample(X_all, y_all)
final_cat_features = [c for c in final_preprocessor.categorical_features_ if c in X_all.columns]
final_params = LGB_PARAMS.copy()
final_params.pop('n_estimators')
prediction_rows = []
final_models = []
for model_no, ((X_sub, y_sub), rounds) in enumerate(zip(final_subsets, best_rounds), start=1):
    print_sampling_summary(y_all, y_sub, f'最终 Easy Ensemble 子模型{model_no}')
    dtrain = lgb.Dataset(X_sub, label=y_sub, feature_name=list(X_sub.columns), categorical_feature=final_cat_features)
    model = lgb.train(final_params, dtrain, num_boost_round=int(rounds), verbose_eval=False)
    final_models.append(model)
    prediction_rows.append(model.predict(X_whitelist, num_iteration=int(rounds)))

predicted_probability = np.mean(np.vstack(prediction_rows), axis=0)
if not np.isfinite(predicted_probability).all():
    raise ValueError('预测概率出现 NaN 或无穷值。')
prediction_list = pd.DataFrame({
    'cst_id': customer_id.astype(str).to_numpy(),
    'predicted_probability': predicted_probability,
}).sort_values('predicted_probability', ascending=False, kind='mergesort').reset_index(drop=True)
prediction_list.to_csv(OUTPUT_FILE, index=False, encoding=CSV_ENCODING)
print(f'预测完成：{len(prediction_list):,} 位客户')
print(f'概率范围：{prediction_list.predicted_probability.min():.6f} ~ {prediction_list.predicted_probability.max():.6f}')
print(f'结果已保存：{OUTPUT_FILE.resolve()}')
prediction_list.head(20)